In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, RANDOM_STATE
from src.cv import load_fold_table
from src.evaluate import cross_validate_segments
from src.features import FEATURES, SIZE_FEATURES, SHAPE_FEATURES

feat = pd.read_csv(PROCESSED / "features_dev.csv")
table = load_fold_table("cv_folds_allclass.csv")
data = feat[feat["y"].isin([1, 2, 3])].reset_index(drop=True)      # seizure vs the two depth-electrode classes
assert set(data["segment"]) <= set(table["segment"])

fold0 = data["segment"].map(table.set_index("segment")["repeat_0"])
print("segments:", len(data), data["y"].value_counts().sort_index().to_dict())
print("segments per fold:", sorted(fold0.value_counts().unique()),
      "| seizure per fold:", sorted(data[data["y"] == 1].groupby(fold0).size().unique()))

segments: 240 {1: 80, 2: 80, 3: 80}
segments per fold: [np.int64(23), np.int64(24), np.int64(25)] | seizure per fold: [np.int64(8)]


In [2]:
feature_sets = {"all 23 features": FEATURES, "size only (8)": SIZE_FEATURES, "shape only (15)": SHAPE_FEATURES}
models = {
    "chance": lambda: DummyClassifier(strategy="prior"),
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random forest": lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=-1),
}
results = {}
for fname, cols in feature_sets.items():
    for mname, make_model in models.items():
        if mname == "chance" and fname != "all 23 features":
            continue                                    # chance ignores the features, so one row is enough
        results[(fname, mname)] = cross_validate_segments(make_model, data, cols, table, positive=1)

summary = pd.DataFrame({k: v.drop(columns="repeat").mean() for k, v in results.items()}).T
summary["auc_sd"] = pd.Series({k: v["auc_fold"].std() for k, v in results.items()})
summary[["auc_fold", "auc_sd", "pr_auc_fold", "sensitivity", "specificity", "precision", "f1", "balanced_accuracy"]].round(3)

auc_fold  auc_sd  pr_auc_fold  \
all 23 features chance                  0.500   0.000        0.333   
                logistic regression     0.997   0.002        0.995   
                random forest           0.999   0.001        0.998   
size only (8)   logistic regression     0.996   0.002        0.993   
                random forest           0.996   0.001        0.988   
shape only (15) logistic regression     0.994   0.003        0.990   
                random forest           0.993   0.003        0.989   

                                     sensitivity  specificity  precision  \
all 23 features chance                     0.000        1.000      0.000   
                logistic regression        0.988        0.988      0.975   
                random forest              0.975        0.981      0.963   
size only (8)   logistic regression        0.978        0.982      0.965   
                random forest              0.975        0.981      0.963   
shape only (15) logistic regression        0.940        0.979      0.957   
                random forest              0.950        0.973      0.945   

                                        f1  balanced_accuracy  
all 23 features chance               0.000              0.500  
                logistic regression  0.981              0.988  
                random forest        0.969              0.978  
size only (8)   logistic regression  0.971              0.980  
                random forest        0.969              0.978  
shape only (15) logistic regression  0.948              0.959  
                random forest        0.948              0.961

In [3]:
group_class = data.groupby("group")["y"].agg(lambda s: s.value_counts().idxmax())
group_size = data.groupby("group").size()
n_seizure = (data["y"] == 1).sum()
logistic = models["logistic regression"]

chance, s = [], 0
while len(chance) < 30:
    rng = np.random.default_rng(100 + s)
    s += 1
    shuffled = pd.Series(rng.permutation(group_class.values), index=group_class.index)    # labels shuffled between groups
    if abs(group_size[shuffled == 1].sum() - n_seizure) > 4:       # keep the seizure count about right
        continue
    d = data.copy()
    d["y"] = d["group"].map(shuffled)
    chance.append(cross_validate_segments(logistic, d, FEATURES, table, n_repeats=1, positive=1)["auc_fold"].iloc[0])
print("AUC with shuffled labels, 30 draws: 5th to 95th percentile", np.round(np.percentile(chance, [5, 95]), 2),
      "| mean", round(np.mean(chance), 3))

AUC with shuffled labels, 30 draws: 5th to 95th percentile [0.35 0.59] | mean 0.487
